# Atividade: Deep Q-Learning com Jogos de Atari 👾 usando RL Baselines3 Zoo

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit4/thumbnail.jpg" alt="Unit 3 Thumbnail">

Neste notebook, **você treinará um agente de Deep Q-Learning** jogando Space Invaders usando o [RL Baselines3 Zoo](https://github.com/DLR-RM/rl-baselines3-zoo), um framework de treinamento baseado no [Stable-Baselines3](https://stable-baselines3.readthedocs.io/en/master/) que fornece scripts para treinar, avaliar agentes, ajustar hiperparâmetros, plotar resultados e gravar vídeos.

Estamos usando a [integração do RL-Baselines-3 Zoo, uma versão vanilla de Deep Q-Learning](https://stable-baselines3.readthedocs.io/en/master/modules/dqn.html) sem extensões como Double-DQN, Dueling-DQN e Prioritized Experience Replay.

⬇️ Aqui está um exemplo do que **você vai alcançar** ⬇️

In [1]:
%%html
<video controls autoplay><source src="https://huggingface.co/ThomasSimonini/ppo-SpaceInvadersNoFrameskip-v4/resolve/main/replay.mp4" type="video/mp4"></video>

### 🎮 Ambientes:

- [SpaceInvadersNoFrameskip-v4](https://gymnasium.farama.org/environments/atari/space_invaders/)

Você pode ver a diferença entre as versões do Space Invaders aqui 👉 https://gymnasium.farama.org/environments/atari/space_invaders/#variants

### 📚 Biblioteca de RL:

- [RL-Baselines3-Zoo](https://github.com/DLR-RM/rl-baselines3-zoo)

## Objetivos deste notebook 🏆
No final do notebook, você irá:
- Compreender melhor **como o RL Baselines3 Zoo funciona**.
- Conseguir **treinar e avaliar seu agente** com um belo replay em vídeo e uma pontuação de avaliação 🔥.

## Um conselho 💡

Os treinos são longos, mas você não perde nada se fechar o notebook: os modelos ficam salvos na pasta `logs/` (um subdiretório novo por experimento). Para retomar, basta reabrir e continuar.

Digite `!nvidia-smi` para ver se há GPU disponível.

# Instalar o RL-Baselines3 Zoo e suas dependências 📚

No original do curso: `!pip install git+https://github.com/DLR-RM/rl-baselines3-zoo` + `!apt-get install swig cmake ffmpeg`. Aqui instalamos a **versão estável do PyPI** e nenhum pacote de sistema é necessário.

Se você vir `ERROR: pip's dependency resolver does not currently take into account all the packages that are installed.`, **é normal e não é um erro crítico** — há um conflito de versão, mas os pacotes de que precisamos estão instalados.

In [2]:
%pip install -q rl_zoo3 "gymnasium[atari]" opencv-python-headless "autorom[accept-rom-license]" imageio imageio-ffmpeg

# - rl_zoo3: scripts de treino/avaliação do RL Baselines3 Zoo
# - gymnasium[atari]: o gymnasium + ale-py (emulador do Atari)
# - opencv-python-headless: pré-processamento dos quadros (AtariWrapper)
# - autorom[accept-rom-license]: baixador das ROMs, já aceitando a licença
# - imageio + imageio-ffmpeg: gravação do vídeo de replay (record_video)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.7/434.7 kB 11.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.0/79.0 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.0/93.0 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 22.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.18.0 requires huggingface-hub<3.0,

## ROMs dos jogos de Atari 🕹️

Os arquivos de ROM (os jogos) **não vêm no pip** — precisam ser baixados e têm licença. O AutoROM faz os dois. A célula seguinte baixa (~50 MB) e a de verificação confere se o ambiente abre.

In [3]:
!AutoROM --accept-license   # baixa as ROMs e aceita a licença automaticamente

/usr/local/lib/python3.13/dist-packages/AutoROM/AutoROM.py:264: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
AutoROM will download the Atari 2600 ROMs.
They will be installed to:
	/usr/local/lib/python3.13/dist-packages/AutoROM/roms

Existing ROMs will be overwritten.


In [4]:
import ale_py  # registra os ids *-v4 (SpaceInvadersNoFrameskip-v4 etc.) no gymnasium
import gymnasium as gym

env = gym.make("SpaceInvadersNoFrameskip-v4")
print("✓ Ambiente ok:", env.observation_space.shape, "| ações:", env.action_space.n)
env.close()

✓ Ambiente ok: (210, 160, 3) | ações: 6


## Criar um display virtual 🔽 (somente Colab)

No Colab (que não tem tela), é preciso criar uma tela virtual para renderizar o ambiente e gravar o replay em vídeo. A célula abaixo instala as bibliotecas e cria a tela virtual 🖥

In [5]:
# (somente no Colab) descomente para criar a tela virtual:
!apt install python-opengl xvfb && pip install pyvirtualdisplay
from pyvirtualdisplay import Display
virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

## Treinar nosso Agente de Deep Q-Learning para Jogar Space Invaders 👾

Para treinar um agente com o RL-Baselines3-Zoo, precisamos apenas fazer duas coisas:

1. Criar um arquivo de configuração de hiperparâmetros chamado `dqn.yml` (a célula abaixo cria). Modelo:

```
SpaceInvadersNoFrameskip-v4:
  env_wrapper:
    - stable_baselines3.common.atari_wrappers.AtariWrapper
  frame_stack: 4
  policy: 'CnnPolicy'
  n_timesteps: !!float 1e6
  buffer_size: 100000
  learning_rate: !!float 1e-4
  batch_size: 32
  learning_starts: 100000
  target_update_interval: 1000
  train_freq: 4
  gradient_steps: 1
  exploration_fraction: 0.1
  exploration_final_eps: 0.01
  # Se for True, você precisa desativar handle_timeout_termination
  # nos replay_buffer_kwargs
  optimize_memory_usage: False
```

In [6]:
%%writefile dqn.yml
SpaceInvadersNoFrameskip-v4:
  env_wrapper:
    - stable_baselines3.common.atari_wrappers.AtariWrapper
  frame_stack: 4
  policy: 'CnnPolicy'
  n_timesteps: !!float 1e6
  buffer_size: 100000
  learning_rate: !!float 1e-4
  batch_size: 32
  learning_starts: 100000
  target_update_interval: 1000
  train_freq: 4
  gradient_steps: 1
  exploration_fraction: 0.1
  exploration_final_eps: 0.01
  # Se for True, você precisa desativar handle_timeout_termination
  # nos replay_buffer_kwargs
  optimize_memory_usage: False

Writing dqn.yml


Aqui nós vemos que:
- Usamos o `Atari Wrapper` que pré-processa a entrada (redução de quadros, escala de cinza, empilhamento de 4 quadros)
- Usamos `CnnPolicy`, já que usamos camadas convolucionais para processar os quadros
- Treinamos por 1 milhão de `n_timesteps`
- O tamanho da memória (Experience Replay) é 100000 — a quantidade de passos de experiência salvos para re-treinar o agente

💡 Para otimizar hiperparâmetros, o conselho do curso é focar nestes 3: `learning_rate`, `buffer_size` e `batch_size`. Documentação: https://stable-baselines3.readthedocs.io/en/master/modules/dqn.html#parameters

2. Iniciamos o treinamento — os modelos ficam na pasta `logs/` 📁

- `--algo`: o algoritmo
- `-f`: pasta onde salvar os modelos
- `-c`: arquivo de hiperparâmetros
- `--gym-packages ale_py`: registra o ambiente `-v4` (necessário nas versões atuais)
- `-n`: número de timesteps (se omitido, vale o `n_timesteps` do `dqn.yml`: 1 milhão)

In [7]:
# Exercício: preencha os espaços, descomente e execute.
# !python -m rl_zoo3.train --algo ________ --env SpaceInvadersNoFrameskip-v4 \
#     -f _________ -c _________ --gym-packages ale_py

<details><summary><b>Solução</b></summary>

```python
!python -m rl_zoo3.train --algo dqn --env SpaceInvadersNoFrameskip-v4 \
    -f logs/ -c dqn.yml --gym-packages ale_py
```
</details>

## Vamos avaliar nosso agente 👀
- O RL-Baselines3-Zoo fornece o `enjoy.py`, um script Python para avaliar nosso agente. Na maioria das bibliotecas de RL, chamamos o script de avaliação de `enjoy.py`.
- Vamos avaliá-lo por 5000 timesteps 🔥

*(No zoo 2.x, `--folder` aponta para a raiz de `logs/` e o modelo avaliado é o do experimento mais recente.)*

In [ ]:
# Exercício: complete o número de timesteps.
# !python -m rl_zoo3.enjoy --algo dqn --env SpaceInvadersNoFrameskip-v4 \
#     --no-render --n-timesteps _________ --folder logs/ --gym-packages ale_py

<details><summary><b>Solução</b></summary>

```python
!python -m rl_zoo3.enjoy --algo dqn --env SpaceInvadersNoFrameskip-v4 \
    --no-render --n-timesteps 5000 --folder logs/ --gym-packages ale_py
```
</details>

## Gravar um vídeo do replay 🎥

A função `record_video` (adaptada do 0_qlearning.ipynb) carrega o modelo treinado e grava um vídeo do agente jogando.

- `modelo_mais_recente()` encontra o modelo mais novo salvo em `logs/`;
- o ambiente é criado com o **mesmo pré-processamento do treino** (`AtariWrapper` + pilha de 4 quadros + transposição), mas com `render_mode="rgb_array"` para capturar os quadros coloridos do jogo;
- `fps=15` ≈ tempo real do jogo (cada passo de treino pula 4 quadros a 60 fps); use `fps=30` para um replay 2× mais rápido.

In [ ]:
import glob
import numpy as np
import imageio
from pathlib import Path

def modelo_mais_recente():
    "Caminho do modelo treinado mais recente (logs/dqn/*.zip)"
    zips = sorted(Path("logs/dqn").glob("SpaceInvadersNoFrameskip-v4_*/SpaceInvadersNoFrameskip-v4.zip"),
                  key=lambda p: p.stat().st_mtime)
    assert zips, "Nenhum modelo treinado em logs/ — rode a célula de treino antes!"
    return str(zips[-1])

def record_video(model_path, out_directory, n_steps=2000, fps=15):
    """
    Gera um vídeo de replay do agente treinado (adaptado do 0_qlearning.ipynb)
    :param model_path: caminho do modelo .zip treinado (use modelo_mais_recente())
    :param out_directory: arquivo de saída (ex.: "replay.mp4")
    :param n_steps: número máximo de passos a gravar
    :param fps: quadros por segundo no vídeo (15 ≈ tempo real do jogo)
    """
    import ale_py  # registra os ids -v4 no gymnasium
    import gymnasium as gym
    from stable_baselines3 import DQN
    from stable_baselines3.common.atari_wrappers import AtariWrapper
    from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack, VecTransposeImage

    model = DQN.load(model_path)
    # Mesma pilha de pré-processamento do treino, mas renderizando em rgb_array
    env = DummyVecEnv([lambda: AtariWrapper(gym.make("SpaceInvadersNoFrameskip-v4",
                                                     render_mode="rgb_array"))])
    env = VecFrameStack(env, 4)
    env = VecTransposeImage(env)

    images = []
    obs = env.reset()
    for _ in range(n_steps):
        action, _ = model.predict(obs, deterministic=True)  # ação do agente treinado
        obs, reward, dones, info = env.step(action)
        img = env.render()
        if isinstance(img, list):
            img = img[0]
        images.append(np.array(img))
        if dones[0]:
            break
    env.close()
    imageio.mimsave(out_directory, images, fps=fps)
    print(f"Vídeo salvo em {out_directory} ({len(images)} quadros)")

In [ ]:
# Gravar um vídeo do resultado do treino
video_path = "replay.mp4"
record_video(modelo_mais_recente(), video_path, n_steps=2000)

### 🥳

Parabéns! Você acabou de treinar seu primeiro agente de Deep Q-Learning usando o RL-Baselines-3 Zoo.

## Carregar um modelo treinado poderoso 🔥
- A equipe do Stable-Baselines3 enviou **mais de 150 agentes de Deep Reinforcement Learning treinados no Hub**: https://huggingface.co/sb3

Alguns exemplos:
- Asteroids: https://huggingface.co/sb3/dqn-AsteroidsNoFrameskip-v4
- Beam Rider: https://huggingface.co/sb3/dqn-BeamRiderNoFrameskip-v4
- Breakout: https://huggingface.co/sb3/dqn-BreakoutNoFrameskip-v4
- Road Runner: https://huggingface.co/sb3/dqn-RoadRunnerNoFrameskip-v4

Vamos carregar um agente jogando Beam Rider:

In [ ]:
%%html
<video controls autoplay><source src="https://huggingface.co/sb3/dqn-BeamRiderNoFrameskip-v4/resolve/main/replay.mp4" type="video/mp4"></video>

1. Baixamos o modelo usando `rl_zoo3.load_from_hub` e o colocamos numa nova pasta `rl_trained`

In [ ]:
!python -m rl_zoo3.load_from_hub --algo dqn --env BeamRiderNoFrameskip-v4 -orga sb3 -f rl_trained/

2. Vamos avaliá-lo por 5000 timesteps

In [ ]:
!python -m rl_zoo3.enjoy --algo dqn --env BeamRiderNoFrameskip-v4 \
    -n 5000 -f rl_trained/ --no-render --gym-packages ale_py

________________________________________________________________________
Parabéns por terminar estaz atividade! 🥳


### Continue Aprendendo, Continue Incrível 🤗